## 1. SQL


### Brushing up SQL Concepts

Here are the conceps you should brush-up on or learn to have a basic understanding of SQL and complete the assignment:

1. **Basic SQL Queries**  
   - Understanding the `SELECT` statement, filtering with `WHERE`, and limiting results with `LIMIT`.

2. **Sorting and Aggregation**  
   - Using `ORDER BY`, `GROUP BY`, `HAVING` clauses and aggregate functions like `COUNT`, `SUM`, etc.

3. **Joins and Relationships**  
   - Combining data from multiple tables using various joins (e.g., `INNER JOIN`, `LEFT JOIN`, self-joins).

4. **Subqueries and Nested Queries**  
   - Writing queries within queries to filter or compute intermediate results.

5. **Common Table Expressions (CTEs)**  
   - Creating temporary result sets to simplify complex queries.

6. **Window Functions**  
   - Applying functions across a set of rows related to the current row (e.g., `RANK`, `ROW_NUMBER`).

7. **Transactions and Data Manipulation**  
   - Using `UPDATE`, `DELETE`, `INSERT`, and managing transactions with commands like `BEGIN` and `COMMIT`.

8. **String Aggregation**  
   - Techniques to combine multiple row values into a single string (e.g., using `GROUP_CONCAT`).


### Resources
I find the best way to learn is to start with looking up a specific concept and then playing around with it using an actual database (can use the Open Flights DB below). It can be also be helpful to use these interactive SQL learning tools:

- SQL ZOO : https://sqlzoo.net/wiki/SQL_Tutorial
- SQL Tutorial : https://www.w3schools.com/sql/

Here is a website for comprehensive sqlite syntax tutorials: https://www.sqlitetutorial.net/

If you want to build a more foundational knowlegde in SQL and then play with the queries:
- cannot go wrong with freecodecamp: https://www.youtube.com/watch?v=HXV3zeQKqGY

### Setting Up sqlite DB

We are going to be using the open flights database. The following code will download the data and then set up a sqllite DB that will be the basis of the quiz. Feel free to play around with using SQL queries

In [1]:
import sqlite3
import pandas as pd

url_dict = {
'airports' : ('https://raw.githubusercontent.com/jpatokal/openflights/master/data/airports.dat',["Airport ID", "Name","City",'Country','IATA','ICAO','Latitude','Longitude','Altitude',
                                                                                                 'Timezone','DST','Tz database timezone','Type','Source']),
'airlines' : ('https://raw.githubusercontent.com/jpatokal/openflights/master/data/airlines.dat',['Airline ID','Name','Alias','IATA','ICAO','Callsign','Country','Active']),
'routes' : ('https://raw.githubusercontent.com/jpatokal/openflights/master/data/routes.dat',['Airline','Airline ID','Source airport','Source airport ID','Destination airport',
                                                                                             'Destination airport ID','Codeshare','Stops','Equipment']),
'planes' : ('https://raw.githubusercontent.com/jpatokal/openflights/master/data/planes.dat',['Name','IATA code','ICAO code'])
}



conn = sqlite3.connect("openflights.db")
cursor = conn.cursor()

for db_name, (url,columns) in url_dict.items():
    df = pd.read_csv(url,names=columns)
    df.columns = df.columns.str.lower().str.replace(" ", "_")
    print(df)
    df.to_sql(db_name,conn,if_exists='replace',index = False),

      airport_id                                         name          city  \
0              1                               Goroka Airport        Goroka   
1              2                               Madang Airport        Madang   
2              3                 Mount Hagen Kagamuga Airport   Mount Hagen   
3              4                               Nadzab Airport        Nadzab   
4              5  Port Moresby Jacksons International Airport  Port Moresby   
...          ...                                          ...           ...   
7693       14106                          Rogachyovo Air Base        Belaya   
7694       14107                        Ulan-Ude East Airport      Ulan Ude   
7695       14108                         Krechevitsy Air Base      Novgorod   
7696       14109                  Desierto de Atacama Airport       Copiapo   
7697       14110                           Melitopol Air Base     Melitopol   

               country iata  icao   latitude   long

In [2]:
#use multiline queries to make it easier to read nad build
query = """
select * from airports
where airport_id = 1
"""

df = pd.read_sql_query(query,conn)
df

,airport_id,name,city,country,iata,icao,latitude,longitude,altitude,timezone,dst,tz_database_timezone,type,source
0,1,Goroka Airport,Goroka,Papua New Guinea,GKA,AYGA,-6.08169,145.391998,5282,10,U,Pacific/Port_Moresby,airport,OurAirports


**Q1. Retrieve all columns for the first 5 rows from the `airports` table.**

In [3]:
query = """
SELECT * FROM airports
LIMIT 5;
"""

df = pd.read_sql_query(query,conn)
df

,airport_id,name,city,country,iata,icao,latitude,longitude,altitude,timezone,dst,tz_database_timezone,type,source
0,1,Goroka Airport,Goroka,Papua New Guinea,GKA,AYGA,-6.081690,145.391998,5282,10,U,Pacific/Port_Moresby,airport,OurAirports
1,2,Madang Airport,Madang,Papua New Guinea,MAG,AYMD,-5.207080,145.789001,20,10,U,Pacific/Port_Moresby,airport,OurAirports
2,3,Mount Hagen Kagamuga Airport,Mount Hagen,Papua New Guinea,HGU,AYMH,-5.826790,144.296005,5388,10,U,Pacific/Port_Moresby,airport,OurAirports
3,4,Nadzab Airport,Nadzab,Papua New Guinea,LAE,AYNZ,-6.569803,146.725977,239,10,U,Pacific/Port_Moresby,airport,OurAirports
4,5,Port Moresby Jacksons International Airport,Port Moresby,Papua New Guinea,POM,AYPY,-9.443380,147.220001,146,10,U,Pacific/Port_Moresby,airport,OurAirports


**Q2. Select the `name`, `city`, and `country` columns from the `airports` table for airports located in the United States.**

In [4]:
query = """
SELECT name, city, country
FROM airports
WHERE country = 'United States';
"""

df = pd.read_sql_query(query,conn)
df

,name,city,country
0,Barter Island LRRS Airport,Barter Island,United States
1,Wainwright Air Station,Fort Wainwright,United States
2,Cape Lisburne LRRS Airport,Cape Lisburne,United States
3,Point Lay LRRS Airport,Point Lay,United States
4,Hilo International Airport,Hilo,United States
...,...,...,...
1507,Camp Pendleton MCAS (Munn Field) Airport,Oceanside,United States
1508,Vidalia Regional Airport,Vidalia,United States
1509,Granbury Regional Airport,Granbury,United States
1510,Oswego County Airport,Fulton,United States


**Q3. (Slight Challenge) Find all active airlines (where `active = 'Y'`) that operate from at least 3 distinct source airports. Display the airline's name, airline_id, and the count of distinct source airports. Order the results by the count (highest first).**

In [5]:
query = """
SELECT al.name, al.airline_id, COUNT(DISTINCT r.source_airport) AS source_airport_count
FROM airlines al
JOIN routes r ON al.airline_id = r.airline_id
WHERE al.active = 'Y'
GROUP BY al.airline_id, al.name
HAVING COUNT(DISTINCT r.source_airport) >= 3
ORDER BY source_airport_count DESC;
"""

df = pd.read_sql_query(query,conn)
df

,name,airline_id,source_airport_count
0,American Airlines,24,429
1,United Airlines,5209,426
2,Air France,137,378
3,KLM Royal Dutch Airlines,3090,360
4,US Airways,5265,348
...,...,...,...
496,Compagnie Africaine d\\'Aviation,11806,3
497,Starline.kz,11963,3
498,BVI Airways,16844,3
499,Asia Wings,17023,3


**Q4. List all airports that serve as a source airport along with their name, city, country, altitude, and the total number of routes departing from that airport. Only include airports with at least one departing route. Order the results by the total number of routes (descending) and then by altitude (descending).**

In [6]:
query = """
SELECT ap.name, ap.city, ap.country, ap.altitude,
       COUNT(r.source_airport) AS total_routes
FROM airports ap
JOIN routes r ON ap.iata = r.source_airport
GROUP BY ap.airport_id
HAVING total_routes >= 1
ORDER BY total_routes DESC, ap.altitude DESC;
"""

df = pd.read_sql_query(query,conn)
df

,name,city,country,altitude,total_routes
0,Hartsfield Jackson Atlanta International Airport,Atlanta,United States,1026,915
1,Chicago O'Hare International Airport,Chicago,United States,672,558
2,Beijing Capital International Airport,Beijing,China,116,535
3,London Heathrow Airport,London,United Kingdom,83,527
4,Charles de Gaulle International Airport,Paris,France,392,524
...,...,...,...,...,...
3247,Hydaburg Seaplane Base,Hydaburg,United States,0,1
3248,Charlotte Amalie Harbor Seaplane Base,Charlotte Amalie,Virgin Islands,0,1
3249,Ulaangom Airport,Ulaangom,Mongolia,0,1
3250,Choiseul Bay Airport,Choiseul Bay,Solomon Islands,0,1


**Q5. List each source airport along with the count of routes departing from it and its rank based on the route count (with the highest count ranked 1). Only include airports with at least 5 routes.**

In [7]:
query = """
SELECT source_airport,
       COUNT(*) AS route_count,
       RANK() OVER (ORDER BY COUNT(*) DESC) AS route_rank
FROM routes
GROUP BY source_airport
HAVING route_count >= 5;
"""

df = pd.read_sql_query(query,conn)
df

,source_airport,route_count,route_rank
0,ATL,915,1
1,ORD,558,2
2,PEK,535,3
3,LHR,527,4
4,CDG,524,5
...,...,...,...
1504,ARI,5,1343
1505,APW,5,1343
1506,AOR,5,1343
1507,AOJ,5,1343


**Q6. (Slight Challenge) List each route's details including the airline name and airline country, the source airport name and city, and the destination airport name and city. Only include routes where both the source and destination airports are located in the same country as the airline.**

In [8]:
query = """
SELECT al.name AS airline_name,
       al.country AS airline_country,
       src.name AS source_airport_name,
       src.city AS source_city,
       dst.name AS destination_airport_name,
       dst.city AS destination_city
FROM routes r
JOIN airlines al  ON r.airline_id     = al.airline_id
JOIN airports src ON r.source_airport = src.iata
JOIN airports dst ON r.destination_airport = dst.iata
WHERE src.country = al.country
  AND dst.country = al.country;
"""

df = pd.read_sql_query(query,conn)
df

,airline_name,airline_country,source_airport_name,source_city,destination_airport_name,destination_city
0,Star Peru (2I),Peru,Coronel FAP Alfredo Mendivil Duarte Airport,Ayacucho,Jorge Chávez International Airport,Lima
1,Star Peru (2I),Peru,Alejandro Velasco Astete International Airport,Cuzco,Jorge Chávez International Airport,Lima
2,Star Peru (2I),Peru,Alejandro Velasco Astete International Airport,Cuzco,Padre Aldamiz International Airport,Puerto Maldonado
3,Star Peru (2I),Peru,Alferez Fap David Figueroa Fernandini Airport,Huánuco,Jorge Chávez International Airport,Lima
4,Star Peru (2I),Peru,Coronel FAP Francisco Secada Vignetta Internat...,Iquitos,Cap FAP David Abenzur Rengifo International Ai...,Pucallpa
...,...,...,...,...,...,...
25351,Regional Express,Australia,Wagga Wagga City Airport,Wagga Wagga,Melbourne International Airport,Melbourne
25352,Regional Express,Australia,Wagga Wagga City Airport,Wagga Wagga,Sydney Kingsford Smith International Airport,Sydney
25353,Regional Express,Australia,Winton Airport,Winton,Longreach Airport,Longreach
25354,Regional Express,Australia,Winton Airport,Winton,Townsville Airport,Townsville


**Q7. Find all airports that do not appear in the routes table at all (neither as a source nor as a destination). List the airport's name, city, and country.**

In [9]:
query = """
SELECT name, city, country
FROM airports
WHERE iata NOT IN (SELECT DISTINCT source_airport      FROM routes WHERE source_airport      IS NOT NULL)
  AND iata NOT IN (SELECT DISTINCT destination_airport FROM routes WHERE destination_airport IS NOT NULL);
"""

df = pd.read_sql_query(query,conn)
df

,name,city,country
0,Hornafjörður Airport,Hofn,Iceland
1,Húsavík Airport,Husavik,Iceland
2,Patreksfjörður Airport,Patreksfjordur,Iceland
3,Siglufjörður Airport,Siglufjordur,Iceland
4,Vestmannaeyjar Airport,Vestmannaeyjar,Iceland
...,...,...,...
4431,Kubinka Air Base,Kubinka,Russia
4432,Rogachyovo Air Base,Belaya,Russia
4433,Ulan-Ude East Airport,Ulan Ude,Russia
4434,Krechevitsy Air Base,Novgorod,Russia


**Q8. List each source airport once along with a comma-separated list of all its distinct destination airports. Only include source airports that have multiple distinct destination airports.**

In [10]:
query = """
SELECT source_airport,
       GROUP_CONCAT(DISTINCT destination_airport) AS destinations
FROM routes
GROUP BY source_airport
HAVING COUNT(DISTINCT destination_airport) > 1;
"""

df = pd.read_sql_query(query,conn)
df

,source_airport,destinations
0,AAE,"ALG,CDG,IST,LYS,MRS,ORN,ORY"
1,AAL,"AMS,AAR,OSL,BLL,SVG,AGP,ALC,CPH,LGW,PMI,BCN,AR..."
2,AAN,"CCJ,PEW"
3,AAQ,"DME,LED,SVO"
4,AAR,"AAL,BMA,GOT,OSL,AGP,PMI,STN,CPH"
...,...,...
2498,ZTH,"ATH,DUS,MUC,VIE,MAN,CRL,EFL,KIT,BRU,AMS,LBA,LGW"
2499,ZUH,"CZX,HAK,PVG,SHA,SYX,TSN,CKG,CTU,FOC,HYN,KHN,KW..."
2500,ZUM,"YWK,YYR"
2501,ZVK,"BKK,PKZ,VTE"


**Q9. Update the `active` status to `'N'` for the airline with a specific `airline_id` (e.g., `1234`) in the `airlines` table.**

In [12]:
cursor.execute("UPDATE airlines SET active = 'N' WHERE airline_id = 1234")
conn.commit()

query = "SELECT airline_id, name, active FROM airlines WHERE airline_id = 1234"
df = pd.read_sql_query(query, conn)
df

,airline_id,name,active
0,1234,Arhabaev Tourism Airlines,N


**Q10. Create a SQL transaction that deletes all routes from the `routes` table where `stops` is greater than 0, commits the transaction, and then verifies that no routes with `stops` greater than 0 remain.**

In [ ]:
# Step 1: Begin transaction and delete
cursor.execute("BEGIN")
cursor.execute("DELETE FROM routes WHERE stops > 0")
conn.commit()

# Step 2: Verify - should return 0
query = """
SELECT COUNT(*) AS remaining_routes_with_stops
FROM routes
WHERE stops > 0
"""
df = pd.read_sql_query(query, conn)
df

,remaining_routes_with_stops
0,0


### SQL Quiz

*All questions require SQL queries, statements, or transactions. Provide your answers as SQL code.*

1. Retrieve all columns for the first 5 rows from the `airports` table.
2. Select the `name`, `city`, and `country` columns from the `airports` table for airports located in the United States.
3. (Slight Challenge) Find all active airlines (where `active = 'Y'`) that operate from at least 3 distinct source airports. Display the airline's name, airline_id, and the count of distinct source airports. Order the results by the count (highest first). 
4. List all airports that serve as a source airport along with their name, city, country, altitude, and the total number of routes departing from that airport. Only include airports with at least one departing route. Order the results by the total number of routes (descending) and then by altitude (descending).
5. List each source airport along with the count of routes departing from it and its rank based on the route count (with the highest count ranked 1). Only include airports with at least 5 routes.
6. (Slight Challenge) List each route's details including the airline name and airline country, the source airport name and city, and the destination airport name and city. Only include routes where both the source and destination airports are located in the same country as the airline.
7. Find all airports that do not appear in the routes table at all (neither as a source nor as a destination). List the airport's name, city, and country.
8. List each source airport once along with a comma-separated list of all its distinct destination airports. Only include source airports that have multiple distinct destination airports.
9. Update the `active` status to `'N'` for the airline with a specific `airline_id` (e.g., `1234`) in the `airlines` table.
10. Create a SQL transaction that deletes all routes from the `routes` table where `stops` is greater than 0, commits the transaction, and then verifies that no routes with `stops` greater than 0 remain.

## 2. Workflow for Assignment Submission

You will create your own repository on GitHub and then upload your assignments by directly pushing to the repo. This approach avoids merge conflicts with files such as IPython Notebook files and keeps your work isolated.

### Workflow Overview

1. Create a Repo On Github  
2. Clone Your Repo Locally  
3. Set Up the Upstream Remote (`git clone` will also do this, just verify)
4. Add any changes and commit
5. Pull any changes in the repo not in local copy (`git pull --rebase`)
6. Push changes

---

### 1. Make your own GitHib Repo

Name it what you like. data-alchemy-assignments doesn't sound too bad 

---

### 2. Clone Your Repo Locally

Open your terminal and run:

`git clone https://github.com/your-username/your-repo.git`

Then change into the repository directory:

`cd your-repo`

---

### 3. Set Up the Upstream Remote

Above step should accomplish this

Verify your remotes with:

`git remote -v`

You should see two remotes:
- **origin** pointing to your repo

---

### 4. Work on and Commit Your Assignment

Add the file to your local checkout from my emails.
Edit the assignment files as needed. When you are ready to save your work, add and commit your changes:

`git add .`  
`git commit -m "Completed week2 assignment: brief description of changes"`

---

### 5. Update Your Local Master with Upstream Changes

If you have made any changes in your master branch, you can stash them before pulling. To stash your changes, run:

`git stash`

*Remember, dont push changes to your master as it will create merge conflicts when you try to pull new changes from the upstream (Pratyush's Assignments repo)*


Then, fetch the latest changes from upstream:

`git fetch upstream`

If you stashed changes earlier and want to reapply them after updating master, run:

`git stash pop`

*Note: Stashing ensures that any local modifications on your master branch do not conflict with the updates you are pulling from upstream.*

---

### 6. Push Your Assignment Branch to Your Fork

Push your branch to your fork on GitHub:

`git push origin assignment-week2`

---

### Tips

- **Stashing Local Changes:**  
  Always stash any uncommitted changes in your master branch before fetching and merging upstream updates to avoid merge conflicts.  


## 3. Additional Tasks

1. How would you put your Build Project experience in your resume?

2. List 3-5 datasets that you would choose from for your final project. This is your chance to pick something in a topic that interests you. We will cover how to find datasets in next session.